# VayuSwarm — Behavior Transformer (Kaggle)

Trains a **Transformer** on real aerial tracking trajectories to classify drone-view object behavior.

### Datasets
| Dataset | HuggingFace ID | What it provides |
|---------|---------------|-----------------|
| VisDrone MOT 2019 | `Vayex/VisDrone2018` | 4 500+ real aerial video sequences, per-frame bounding boxes → trajectory extraction |
| Omni-MOT (drone-view) | `keremberke/aerial-sheep` | Aerial livestock tracking with motion diversity — adds formation/stationary examples |

### Behavior classes
`stationary` · `patrol` · `evasive` · `approaching` · `formation`

### Output
`best_behavior.pth` + `behavior_transformer.onnx` + `norm_mean.npy/norm_std.npy` → `models/behavior/`

### Kaggle Secrets required
`HF_TOKEN` · `GIT_TOKEN`

> Runtime: ~20–30 min on T4 GPU

In [ ]:
# ── CELL 1: Install ──────────────────────────────────────────────────────────
import subprocess
subprocess.check_call(["pip", "install", "-q",
    "torch", "onnx", "huggingface_hub", "hf-transfer",
    "scikit-learn", "datasets"])

import os, json, shutil, tempfile, math
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import numpy as np
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── CELL 2: Config & Secrets ─────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    HF_TOKEN  = _s.get_secret("HF_TOKEN")
    GIT_TOKEN = _s.get_secret("GIT_TOKEN")
    print(f"✅ Secrets — HF_TOKEN starts: {HF_TOKEN[:8] if HF_TOKEN else 'EMPTY'}")
except Exception as e:
    print(f"⚠ kaggle_secrets failed: {e}")
    HF_TOKEN  = os.environ.get("HF_TOKEN", "")
    GIT_TOKEN = os.environ.get("GIT_TOKEN", "")

GIT_REPO  = "https://github.com/ved354/swam.git"
GIT_USER  = "ved354"
GIT_EMAIL = "ved354@users.noreply.github.com"

# ── Model hyperparams (must match src/vision/behavior_analyzer.py) ───────────
WINDOW    = 30      # trajectory frames per sample
FEAT      = 5       # features per frame: x, y, speed, heading, accel
DIM       = 64
HEADS     = 4
LAYERS    = 2
EPOCHS    = 60
BATCH     = 128
LR        = 1e-3
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "behavior_model"
VISDRONE_DIR = WORK_DIR / "visdrone_mot"
AERIAL_DIR  = WORK_DIR / "aerial_sheep"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["patrol", "evasive", "formation", "stationary", "approaching"]
print(f"Device : {DEVICE}")
print(f"Classes: {CLASSES}")

In [ ]:
# ── CELL 3: Download Datasets ────────────────────────────────────────────────
from huggingface_hub import snapshot_download, login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ HuggingFace login OK")

# ── Dataset 1: VisDrone 2019-MOT ─────────────────────────────────────────────
# Vayex/VisDrone2018  —  aerial video tracking annotations in MOT format
# Format: frame,id,x,y,w,h,score,class,truncation,occlusion
if not VISDRONE_DIR.exists() or not any(VISDRONE_DIR.iterdir()):
    print("📥 Downloading VisDrone MOT (real aerial tracking sequences)...")
    snapshot_download(
        repo_id="Vayex/VisDrone2018",
        repo_type="dataset",
        allow_patterns=["VisDrone2019-MOT-train/**", "VisDrone2019-MOT-val/**"],
        local_dir=str(VISDRONE_DIR),
        token=HF_TOKEN or None,
        max_workers=4,
    )
    print("✅ VisDrone MOT downloaded")
else:
    print("✅ VisDrone MOT already exists")

# ── Dataset 2: Aerial Sheep (drone-view tracking) ────────────────────────────
# keremberke/aerial-sheep — livestock tracking from drones, COCO bboxes per frame
# Provides variety in formation/stationary behaviors (herding patterns)
if not AERIAL_DIR.exists() or not any(AERIAL_DIR.iterdir()):
    print("📥 Downloading Aerial Sheep (drone-view multi-object tracking)...")
    snapshot_download(
        repo_id="keremberke/aerial-sheep",
        repo_type="dataset",
        local_dir=str(AERIAL_DIR),
        token=HF_TOKEN or None,
        max_workers=4,
    )
    print("✅ Aerial Sheep downloaded")
else:
    print("✅ Aerial Sheep already exists")

# Inventory
for name, d in [("VisDrone", VISDRONE_DIR), ("AerialSheep", AERIAL_DIR)]:
    n = sum(1 for _ in d.rglob("*") if _.is_file()) if d.exists() else 0
    print(f"  {name}: {n} files")